# Train NEXUS Command Classifiers (Tier 3 — Skip ASR for Known Commands)

This notebook trains **multiple** openWakeWord command classifiers in a single Colab session.
Each model detects a spoken command like "open youtube" or "open gmail" directly from audio —
**no ASR needed**. When a command model fires, NEXUS executes the action in ~200ms instead of ~30s.

**Based on**: `train_nexus_oww.ipynb` (the wake-word trainer). This notebook reuses the same
infrastructure (Piper TTS, FMA noise, ACAV100M negatives, MIT RIRs) but trains a **loop** of commands.

**Runtime**: ~4-6 hours on Colab Pro (L4 GPU + High RAM) for 10 commands.
  - Setup (install + downloads): ~30 min (done ONCE)
  - Per command (generate clips + augment + train + export): ~15-25 min

**Output**: Multiple `.onnx` files (one per command), each ~800KB.

## Commands to train
Edit the `COMMANDS` list in the config cell to choose which commands to train.
Each command needs:
  - `phrase`: what the user says (e.g. "open youtube")
  - `model_name`: filename for the ONNX model (e.g. "open_youtube")
  - `negatives`: similar-sounding phrases to exclude (reduces false positives)
  - `intent`: the structured intent to emit when detected

## Instructions
1. **Runtime → Change runtime type → T4 GPU** (or L4 GPU + High RAM on Colab Pro)
2. **Runtime → Run all**
3. Each model is auto-downloaded as it completes. Place all `.onnx` files in:
   `src-tauri/resources/oww/commands/`

## 1. Install deps (same as wake-word notebook)

In [ ]:
# Native deps
!apt-get install -y -qq cmake espeak-ng espeak-ng-data libespeak-ng-dev libsndfile1 pkg-config build-essential ffmpeg unzip 2>&1 | tail -3

# Python deps — same order as wake-word notebook
!pip install -q piper-phonemize-cross
!pip install -q \
    webrtcvad \
    mutagen==1.47.0 \
    torchinfo \
    torchmetrics \
    pyyaml \
    tqdm \
    datasets \
    soundfile \
    audiomentations \
    torch_audiomentations \
    pronouncing \
    onnxruntime \
    onnx \
    speechbrain \
    acoustics \
    scipy \
    requests \
    huggingface_hub
!pip install -q --no-deps piper-tts

import sys
print('Python:', sys.version)
import torch, torchinfo, torchmetrics, scipy, numpy
print(f'  torch: {torch.__version__}  cuda: {torch.cuda.is_available()}')
from tqdm import tqdm
import yaml, mutagen, pronouncing
import torchaudio, audiomentations, torch_audiomentations
import speechbrain, acoustics
import onnx, onnxruntime, soundfile, requests
from piper_phonemize import phonemize_espeak
from piper import PiperVoice, SynthesisConfig
print('  All deps import cleanly.')

## 2. Clone repos + download shared models (done ONCE)

In [ ]:
import os, sys
os.chdir('/content')

# piper-sample-generator (pinned to flat-layout commit)
PSG_DIR = '/content/piper-sample-generator'
PSG_PIN = '1a8c49bd29b3a132721086ee88f2253f788594a8^'
PSG_GS  = f'{PSG_DIR}/generate_samples.py'
if not os.path.exists(PSG_GS):
    !rm -rf {PSG_DIR}
    !git clone -q https://github.com/rhasspy/piper-sample-generator {PSG_DIR}
!cd {PSG_DIR} && git fetch -q --all && git checkout -q {PSG_PIN}
assert os.path.exists(PSG_GS), 'piper-sample-generator pin failed'
print(f'  OK piper-sample-generator')

# libritts model (~200 MB)
PIPER_MODEL = f'{PSG_DIR}/models/en_US-libritts_r-medium.pt'
PIPER_MODEL_URL = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
if not os.path.exists(PIPER_MODEL) or os.path.getsize(PIPER_MODEL) < 100_000_000:
    !mkdir -p {PSG_DIR}/models
    !wget -q --tries=5 --timeout=300 -O {PIPER_MODEL} {PIPER_MODEL_URL}
assert os.path.getsize(PIPER_MODEL) > 100_000_000
print(f'  OK libritts model: {os.path.getsize(PIPER_MODEL)/1e6:.0f} MB')

# openwakeword
OWW_DIR = '/content/openwakeword'
OWW_TRAIN = f'{OWW_DIR}/openwakeword/train.py'
if not os.path.exists(OWW_TRAIN):
    !rm -rf {OWW_DIR}
    !git clone -q https://github.com/dscripka/openwakeword {OWW_DIR}
    !pip install -q -e {OWW_DIR}
assert os.path.exists(OWW_TRAIN)
if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)
for _m in list(sys.modules):
    if _m.startswith('openwakeword'):
        del sys.modules[_m]
import openwakeword
assert openwakeword.__file__ is not None
print(f'  OK openwakeword: {openwakeword.__file__}')

## 3. Apply runtime patches (same 6 patches as wake-word notebook)

In [ ]:
import os, glob

# Patch A: torchaudio.set_audio_backend → pass
for path in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    !sed -i 's|torchaudio.set_audio_backend("soundfile")|pass  # patched|' "{path}"

# Patch B: copy generate_samples.py
src = '/content/piper-sample-generator/generate_samples.py'
dst = '/content/openwakeword/openwakeword/generate_samples.py'
if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
    !cp "{src}" "{dst}"

# Patch C: HF Hub timeouts
os.environ['HF_HUB_ETAG_TIMEOUT'] = '120'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
import huggingface_hub.constants as hfc
for attr in ['DEFAULT_ETAG_TIMEOUT', 'DEFAULT_DOWNLOAD_TIMEOUT',
             'HF_HUB_ETAG_TIMEOUT', 'HF_HUB_DOWNLOAD_TIMEOUT']:
    if hasattr(hfc, attr): setattr(hfc, attr, 120)

# Patch D: torchaudio.info shim
import torchaudio
init_path = torchaudio.__file__
SHIM_MARKER = '# PATCH: info() shim for torchaudio 2.x'
with open(init_path) as f:
    content = f.read()
if SHIM_MARKER not in content:
    shim = (f'\n\n{SHIM_MARKER}\n'
            'def info(file_path, *args, **kwargs):\n'
            '    import soundfile as _sf\n'
            '    si = _sf.info(str(file_path))\n'
            '    return type("_TorchaudioInfo", (), {\n'
            '        "num_frames": si.frames, "sample_rate": si.samplerate,\n'
            '        "num_channels": si.channels, "bits_per_sample": 16,\n'
            '        "encoding": "PCM_S",\n'
            '    })()\n')
    with open(init_path, 'a') as f: f.write(shim)
    import importlib; importlib.reload(torchaudio)

# Patch E: generate_samples model arg default
TARGET = '/content/piper-sample-generator/generate_samples.py'
!sed -i 's|model: Union\[str, Path\],|model: Union[str, Path] = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",|' "{TARGET}"
!cp "{TARGET}" /content/openwakeword/openwakeword/generate_samples.py

# Patch F: train.py val dtype cast
TRAIN_PY = '/content/openwakeword/openwakeword/train.py'
!sed -i 's|val_predictions = self.model(x_val)$|val_predictions = self.model(x_val.float())|' "{TRAIN_PY}"

print('All 6 patches applied.')

## 4. Download shared data (MIT RIRs, FMA, ACAV100M — done ONCE)

In [ ]:
import os

# Shared OWW models (melspectrogram + embedding)
OWW_MODELS_DIR = '/content/oww_models'
os.makedirs(OWW_MODELS_DIR, exist_ok=True)
for fname in ['melspectrogram.onnx', 'embedding_model.onnx']:
    path = f'{OWW_MODELS_DIR}/{fname}'
    if not os.path.exists(path):
        url = f'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{fname}'
        !wget -q --tries=5 -O {path} {url}
    print(f'  OK {fname}: {os.path.getsize(path)/1e6:.1f} MB')

# MIT RIRs
RIR_DIR = '/content/mit_rirs'
if not os.path.exists(RIR_DIR) or len(os.listdir(RIR_DIR)) < 250:
    from huggingface_hub import snapshot_download
    for attempt in range(6):
        try:
            snapshot_download('davidscripka/MIT_environmental_impulse_responses',
                              repo_type='dataset', local_dir='/content/mit_rirs_raw')
            break
        except Exception as e:
            print(f'  retry {attempt+1}: {e}')
    os.makedirs(RIR_DIR, exist_ok=True)
    import soundfile as sf, glob
    from scipy.signal import resample_poly
    for f in glob.glob('/content/mit_rirs_raw/**/*.wav', recursive=True):
        data, sr = sf.read(f)
        if sr != 16000:
            data = resample_poly(data.astype('float32'), 16000, sr)
        sf.write(f'{RIR_DIR}/{os.path.basename(f)}', data, 16000)
print(f'  OK MIT RIRs: {len(os.listdir(RIR_DIR))} files')

# FMA small dataset (~8 GB)
FMA_ZIP = '/content/fma_small.zip'
FMA_DIR = '/content/fma'
FMA_WAV = '/content/fma_wav'
if not os.path.exists(FMA_DIR):
    if not os.path.exists(FMA_ZIP):
        !wget -q --tries=5 -O {FMA_ZIP} https://os.unil.cloud.switch.ch/fma/fma_small.zip
    !unzip -q -o {FMA_ZIP} -d /content/
print(f'  OK FMA: {os.path.exists(FMA_DIR)}')

# Convert FMA MP3s to WAVs (1500 clips)
if not os.path.exists(FMA_WAV) or len(os.listdir(FMA_WAV)) < 1000:
    os.makedirs(FMA_WAV, exist_ok=True)
    import glob, subprocess
    mp3s = sorted(glob.glob(f'{FMA_DIR}/**/*.mp3', recursive=True))[:1500]
    for mp3 in mp3s:
        wav = f'{FMA_WAV}/{os.path.splitext(os.path.basename(mp3))[0]}.wav'
        if not os.path.exists(wav):
            subprocess.run(['ffmpeg', '-y', '-i', mp3, '-ar', '16000',
                           '-ac', '1', '-t', '30', wav],
                          capture_output=True)
print(f'  OK FMA WAVs: {len(os.listdir(FMA_WAV))} files')

# ACAV100M features (~17 GB)
ACAV_SRC = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
ACAV_TRAIN = '/content/acav_train_subset.npy'
ACAV_VAL = '/content/acav_val_subset.npy'
if not os.path.exists(ACAV_SRC):
    !wget -q --tries=5 --timeout=600 -O {ACAV_SRC} \
        'https://huggingface.co/datasets/dscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
if not os.path.exists(ACAV_TRAIN):
    import numpy as np
    full = np.load(ACAV_SRC, mmap_mode='r')
    train_chunk = full[:len(full)//10]
    np.save(ACAV_TRAIN, train_chunk)
    val_chunk = full[len(full)//10:len(full)//10 + len(full)//100]
    val_flat = val_chunk.reshape(-1, val_chunk.shape[-1])
    np.save(ACAV_VAL, val_flat)
    del full, train_chunk, val_chunk
    os.remove(ACAV_SRC)
print(f'  OK ACAV train: {os.path.getsize(ACAV_TRAIN)/1e9:.1f} GB')
print(f'  OK ACAV val: {os.path.getsize(ACAV_VAL)/1e6:.0f} MB')
print('\nAll shared data ready.')

## 5. Command configuration — EDIT THIS TO CHANGE COMMANDS

Each command trains a separate OWW classifier. The shared infrastructure (Piper TTS, FMA, ACAV, RIRs) is reused across all commands.

Add or remove commands from this list. Each command trains in ~15-25 min.

In [ ]:
# ─── COMMANDS TO TRAIN ───────────────────────────────────────────────
# Each entry trains a separate OWW classifier model.
# phrase:     what the user says (used for TTS clip generation)
# model_name: filename for the .onnx output (no spaces, lowercase)
# negatives:  similar-sounding phrases to exclude (reduces false positives)
# intent:     the structured intent NEXUS will execute when detected

COMMANDS = [
    {
        'phrase': 'open youtube',
        'model_name': 'open_youtube',
        'negatives': ['open you tube', 'open utube', 'open youth tube', 'open u tube'],
        'intent': {'action': 'open_app', 'target': 'youtube'},
    },
    {
        'phrase': 'open gmail',
        'model_name': 'open_gmail',
        'negatives': ['open gamail', 'open gee mail', 'open j mail', 'open email'],
        'intent': {'action': 'open_app', 'target': 'gmail'},
    },
    {
        'phrase': 'open chrome',
        'model_name': 'open_chrome',
        'negatives': ['open crowm', 'open comb', 'open chrome book', 'open krom'],
        'intent': {'action': 'open_app', 'target': 'chrome'},
    },
    {
        'phrase': 'open notepad',
        'model_name': 'open_notepad',
        'negatives': ['open note pad', 'open node pad', 'open no pad', 'open notebook'],
        'intent': {'action': 'open_app', 'target': 'notepad'},
    },
    {
        'phrase': 'open calculator',
        'model_name': 'open_calculator',
        'negatives': ['open calc', 'open calculate', 'open kalculator', 'open count'],
        'intent': {'action': 'open_app', 'target': 'calculator'},
    },
    {
        'phrase': 'open spotify',
        'model_name': 'open_spotify',
        'negatives': ['open spot ify', 'open spot a fy', 'open spotty fight', 'open spy'],
        'intent': {'action': 'open_app', 'target': 'spotify'},
    },
    {
        'phrase': 'open discord',
        'model_name': 'open_discord',
        'negatives': ['open this cord', 'open disk cord', 'open dis cord', 'open accord'],
        'intent': {'action': 'open_app', 'target': 'discord'},
    },
    {
        'phrase': 'open github',
        'model_name': 'open_github',
        'negatives': ['open git hub', 'open get hub', 'open gith ub', 'open gift hub'],
        'intent': {'action': 'open_app', 'target': 'github'},
    },
    {
        'phrase': 'open vscode',
        'model_name': 'open_vscode',
        'negatives': ['open vs code', 'open v s code', 'open vcode', 'open code'],
        'intent': {'action': 'open_app', 'target': 'vscode'},
    },
    {
        'phrase': 'open figma',
        'model_name': 'open_figma',
        'negatives': ['open fig ma', 'open fig mma', 'open big ma', 'open sigma'],
        'intent': {'action': 'open_app', 'target': 'figma'},
    },
]

print(f'Commands to train: {len(COMMANDS)}')
for c in COMMANDS:
    print(f'  {c["model_name"]:20s} ← "{c["phrase"]}"  ({len(c["negatives"])} negatives)')
print(f'\nEstimated time: {len(COMMANDS) * 20} min ({len(COMMANDS) * 20 / 60:.1f} hrs)')

## 6. Training loop — trains each command sequentially

For each command:
1. Generate Piper TTS clips (positive = command phrase, negative = adversarial words)
2. Resample 22050 → 16000 Hz
3. Augment + extract features
4. Train DNN classifier (3-stage curriculum)
5. Ensemble + export ONNX
6. Download the `.onnx` file

The shared data (FMA, ACAV, RIRs) is loaded once and reused across all commands.

In [ ]:
import os, sys, copy, math, time, yaml, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.signal import resample_poly
from tqdm.auto import tqdm
import soundfile as sf
import glob

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ─── Load shared ACAV data ONCE ──────────────────────────────────────
acav_train_np = np.load('/content/acav_train_subset.npy', mmap_mode='r')
acav_val_np = np.load('/content/acav_val_subset.npy')
M = acav_val_np.shape[0]
val_listen_hours = M * 0.08 / 3600.0
n_win_val = M - 16
acav_val_windows = np.lib.stride_tricks.sliding_window_view(acav_val_np, (16, 96))[:, 0, :, :]
acav_val_windows = np.ascontiguousarray(acav_val_windows.astype(np.float32))
print(f'  ACAV train (mmap): {acav_train_np.shape}')
print(f'  ACAV val windows: {acav_val_windows.shape} ({acav_val_windows.nbytes/1e6:.0f} MB, {val_listen_hours:.2f} hr)')
VAL_BATCH = 4096

# ─── DNN model (same architecture as wake-word) ──────────────────────
class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)
    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))

# ─── Helper: generate clips for one command ──────────────────────────
def generate_clips_for_command(cmd, n_samples=2000, n_samples_val=1000):
    """Generate Piper TTS clips for a command phrase."""
    model_name = cmd['model_name']
    phrase = cmd['phrase']
    negatives = cmd['negatives']
    output_dir = f'/content/{model_name}_output'

    config = {
        'target_phrase': [phrase],
        'model_name': model_name,
        'custom_negative_phrases': negatives,
        # Also include all OTHER command phrases as negatives
        # (so "open youtube" doesn't trigger "open gmail" model)
        'n_samples': n_samples,
        'n_samples_val': n_samples_val,
        'tts_batch_size': 50,
        'piper_sample_generator_path': '/content/piper-sample-generator',
        'augmentation_rounds': 1,
        'augmentation_batch_size': 16,
        'steps': 20000,
        'max_negative_weight': 1500,
        'target_accuracy': 0.7,
        'target_recall': 0.5,
        'target_false_positives_per_hour': 0.5,
        'batch_size': 128,
        'learning_rate': 1e-4,
        'model_type': 'dnn',
        'layer_dim': 128,
        'layer_size': 128,
        'n_blocks': 1,
        'model_input_shape': [16, 96],
        'n_classes': 1,
        'batch_n_per_class': {
            'ACAV100M_sample': 1024,
            'adversarial_negative': 50,
            'positive': 50,
        },
        'background_paths': ['/content/fma_wav'],
        'background_paths_duplication_rate': [1],
        'rir_paths': ['/content/mit_rirs'],
        'false_positive_validation_data_path': '/content/acav_val_subset.npy',
        'feature_data_files': {'ACAV100M_sample': '/content/acav_train_subset.npy'},
        'output_dir': output_dir,
        'tflite_export': False,
        'onnx_export': True,
        'positive_clips_train_dir': f'{output_dir}/{model_name}/positive_train',
        'positive_clips_test_dir': f'{output_dir}/{model_name}/positive_test',
        'negative_clips_train_dir': f'{output_dir}/{model_name}/negative_train',
        'negative_clips_test_dir': f'{output_dir}/{model_name}/negative_test',
        'feature_save_dir': f'{output_dir}/{model_name}',
    }
    os.makedirs(output_dir, exist_ok=True)
    config_path = f'/content/{model_name}_config.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)
    return config, config_path

# ─── Helper: resample clips ──────────────────────────────────────────
def resample_clips(output_dir):
    TARGET_SR = 16000
    wav_dirs = sorted({os.path.dirname(f)
                       for f in glob.glob(f'{output_dir}/**/*.wav', recursive=True)})
    for d in wav_dirs:
        files = [f for f in os.listdir(d) if f.endswith('.wav')]
        if not files: continue
        sr = sf.info(f'{d}/{files[0]}').samplerate
        if sr == TARGET_SR: continue
        for f in files:
            p = f'{d}/{f}'
            data, sr = sf.read(p)
            if sr != TARGET_SR:
                new_data = resample_poly(data.astype('float32'), TARGET_SR, sr)
                sf.write(p, new_data, TARGET_SR)
    # Clear stale features
    for f in glob.glob(f'{output_dir}/**/*.npy', recursive=True):
        os.remove(f)

# ─── Helper: train one command ───────────────────────────────────────
def train_command(cmd, config, config_path):
    model_name = cmd['model_name']
    FEAT = config['feature_save_dir']

    # Check if already trained
    onnx_path = f'{FEAT}/{model_name}.onnx'
    if os.path.exists(onnx_path):
        print(f'  SKIP {model_name}: already trained ({onnx_path})')
        return onnx_path

    # Generate clips
    dirs = {
        'positive_train': (config['positive_clips_train_dir'], int(config['n_samples'] * 0.75)),
        'positive_test':  (config['positive_clips_test_dir'],  int(config['n_samples_val'] * 0.75)),
        'negative_train': (config['negative_clips_train_dir'], int(config['n_samples'] * 0.75)),
        'negative_test':  (config['negative_clips_test_dir'],  int(config['n_samples_val'] * 0.75)),
    }
    all_full = all(os.path.isdir(p) and len(os.listdir(p)) >= exp for _, (p, exp) in dirs.items())
    if not all_full:
        print(f'  generating clips for "{cmd["phrase"]}"...')
        !{sys.executable} /content/openwakeword/openwakeword/train.py \
            --training_config {config_path} --generate_clips 2>&1 | tail -5

    # Resample
    resample_clips(config['output_dir'])

    # Augment + featurize
    needed = ['positive_features_train.npy', 'negative_features_train.npy',
              'positive_features_test.npy', 'negative_features_test.npy']
    if not all(os.path.exists(f'{FEAT}/{n}') for n in needed):
        print(f'  augmenting + featurizing...')
        !{sys.executable} /content/openwakeword/openwakeword/train.py \
            --training_config {config_path} --augment_clips 2>&1 | tail -5
    for n in needed:
        assert os.path.exists(f'{FEAT}/{n}'), f'augment failed: {n}'

    # Load features
    pos_train = torch.from_numpy(np.load(f'{FEAT}/positive_features_train.npy').astype(np.float32)).to(DEVICE)
    neg_train = torch.from_numpy(np.load(f'{FEAT}/negative_features_train.npy').astype(np.float32)).to(DEVICE)
    pos_test  = torch.from_numpy(np.load(f'{FEAT}/positive_features_test.npy').astype(np.float32)).to(DEVICE)
    neg_test  = torch.from_numpy(np.load(f'{FEAT}/negative_features_test.npy').astype(np.float32)).to(DEVICE)
    print(f'  features: pos_train={tuple(pos_train.shape)} neg_train={tuple(neg_train.shape)}')

    # Train
    model = WakewordModel().to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(reduction='none')
    TOTAL_STEPS = config['steps']
    MAX_NEG_W = config['max_negative_weight']
    TARGET_FP = config['target_false_positives_per_hour']
    THRESH = 0.5
    history = {'val_recall': [], 'val_accuracy': [], 'val_fp_per_hour': [], 'val_n_fp': [], 'loss': []}
    best_models = []

    B_POS, B_ANEG, B_ACAV = 32, 32, 64
    def random_acav_window_batch(k):
        N, T, F = acav_train_np.shape
        rows = np.random.randint(0, N, size=k)
        starts = np.random.randint(0, T - 16 + 1, size=k)
        out = np.empty((k, 16, F), dtype=np.float32)
        for i, (r, s) in enumerate(zip(rows, starts)):
            out[i] = acav_train_np[r, s:s+16, :].astype(np.float32)
        return out

    def build_batch():
        p_idx = torch.randint(0, pos_train.shape[0], (B_POS,), device=DEVICE)
        aneg_idx = torch.randint(0, neg_train.shape[0], (B_ANEG,), device=DEVICE)
        p, an = pos_train[p_idx], neg_train[aneg_idx]
        acav = torch.from_numpy(random_acav_window_batch(B_ACAV)).to(DEVICE)
        x = torch.cat([p, an, acav], dim=0)
        y = torch.cat([torch.ones(B_POS, device=DEVICE),
                       torch.zeros(B_ANEG + B_ACAV, device=DEVICE)])
        return x, y

    @torch.no_grad()
    def validate(label):
        model.eval()
        p_preds = torch.sigmoid(model(pos_test)).squeeze(-1)
        n_preds = torch.sigmoid(model(neg_test)).squeeze(-1)
        recall = (p_preds >= THRESH).float().mean().item()
        accuracy = (((p_preds >= THRESH).sum() + (n_preds < THRESH).sum()).item()
                    / (pos_test.shape[0] + neg_test.shape[0]))
        n_fp = 0
        for i in range(0, n_win_val, VAL_BATCH):
            chunk = torch.from_numpy(acav_val_windows[i:i+VAL_BATCH]).to(DEVICE)
            n_fp += (torch.sigmoid(model(chunk)).squeeze(-1) >= THRESH).sum().item()
        fp_per_hour = n_fp / max(val_listen_hours, 1e-6)
        history['val_recall'].append(recall)
        history['val_accuracy'].append(accuracy)
        history['val_fp_per_hour'].append(fp_per_hour)
        history['val_n_fp'].append(n_fp)
        save = False
        if len(history['val_n_fp']) >= 3:
            fp_p50 = np.percentile(history['val_n_fp'], 50)
            rc_p5 = np.percentile(history['val_recall'], 5)
            if n_fp <= fp_p50 and recall >= rc_p5:
                best_models.append((copy.deepcopy(model.state_dict()),
                                    {'val_recall': recall, 'val_accuracy': accuracy,
                                     'val_fp_per_hour': fp_per_hour, 'val_n_fp': n_fp}))
                save = True
        print(f'    [{label}] recall={recall:.3f} acc={accuracy:.3f} fp/hr={fp_per_hour:.2f} {"+" if save else "-"}', flush=True)
        model.train()

    def run_stage(idx, n_steps, lr, max_neg_w, val_window_frac=1.0):
        optimizer = optim.Adam(model.parameters(), lr=lr)
        weight_schedule = np.linspace(1.0, max_neg_w, n_steps)
        val_start = int(n_steps * (1.0 - val_window_frac))
        val_steps = set(np.linspace(val_start, n_steps - 1, 20).astype(int))
        warmup = max(1, n_steps // 5)
        hold = n_steps // 3
        accumulated = []
        t0 = time.time()
        for step in range(n_steps):
            if step < warmup: lr_now = lr * (step + 1) / warmup
            elif step < warmup + hold: lr_now = lr
            else:
                decay_t = (step - warmup - hold) / max(1, n_steps - warmup - hold)
                lr_now = lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_t)))
            for pg in optimizer.param_groups: pg['lr'] = lr_now
            x, y = build_batch()
            logits = model(x).squeeze(-1)
            preds = torch.sigmoid(logits)
            keep = ((y == 0) & (preds >= 0.001)) | ((y == 1) & (preds < 0.999))
            if keep.sum() == 0:
                if step in val_steps: validate(f's{idx} {step}/{n_steps}')
                continue
            kept_logits = logits[keep]
            kept_y = y[keep]
            neg_w = weight_schedule[step]
            w = torch.where(kept_y > 0.5,
                            torch.tensor(1.0, device=DEVICE),
                            torch.tensor(neg_w, device=DEVICE, dtype=torch.float32))
            accumulated.append((kept_logits, kept_y, w))
            if sum(t[0].shape[0] for t in accumulated) >= 128:
                cat_logits = torch.cat([t[0] for t in accumulated])
                cat_y = torch.cat([t[1] for t in accumulated])
                cat_w = torch.cat([t[2] for t in accumulated])
                loss = (loss_fn(cat_logits, cat_y) * cat_w).mean()
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                history['loss'].append(loss.item())
                accumulated.clear()
            if step in val_steps:
                validate(f's{idx} {step}/{n_steps} ({(time.time()-t0)/60:.1f}m)')

    max_neg_w_now = MAX_NEG_W
    run_stage(1, TOTAL_STEPS, lr=1e-4, max_neg_w=max_neg_w_now, val_window_frac=0.25)
    if history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP:
        max_neg_w_now *= 2
    run_stage(2, max(2000, TOTAL_STEPS // 10), lr=1e-5, max_neg_w=max_neg_w_now)
    if history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP:
        max_neg_w_now *= 2
    run_stage(3, max(2000, TOTAL_STEPS // 10), lr=1e-6, max_neg_w=max_neg_w_now)
    print(f'  training done: {len(best_models)} checkpoints, best fp/hr={min(history["val_fp_per_hour"]):.2f}')

    # Ensemble + export
    if not best_models:
        final_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        accs = [s['val_accuracy'] for _, s in best_models]
        rcs = [s['val_recall'] for _, s in best_models]
        fps = [s['val_fp_per_hour'] for _, s in best_models]
        acc_p90 = np.percentile(accs, 90)
        rc_p90 = np.percentile(rcs, 90)
        fp_p10 = np.percentile(fps, 10)
        qualified = [(sd, sc) for sd, sc in best_models
                     if sc['val_accuracy'] >= acc_p90 and sc['val_recall'] >= rc_p90
                     and sc['val_fp_per_hour'] <= fp_p10]
        if not qualified:
            qualified = [sorted(best_models, key=lambda t: (t[1]['val_fp_per_hour'], -t[1]['val_recall']))[0]]
        keys = qualified[0][0].keys()
        final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    model.load_state_dict(final_state)
    model.eval()

    class WakewordExportable(nn.Module):
        def __init__(self, base): super().__init__(); self.base = base
        def forward(self, x): return torch.sigmoid(self.base(x))
    export_model = WakewordExportable(model).to(DEVICE).eval()
    dummy = torch.randn(1, 16, 96, device=DEVICE)
    torch.onnx.export(export_model, dummy, onnx_path,
                      input_names=['onnx::Flatten_0'], output_names=['output'],
                      dynamic_axes={'onnx::Flatten_0': {0: 'batch'}, 'output': {0: 'batch'}},
                      opset_version=14, dynamo=False)
    print(f'  exported: {onnx_path} ({os.path.getsize(onnx_path)/1e3:.0f} KB)')

    # Sanity check
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path)
    with torch.no_grad():
        p_scores = sess.run(None, {sess.get_inputs()[0].name: pos_test.cpu().numpy()})[0].flatten()
        print(f'  sanity: recall@0.5={(p_scores >= 0.5).mean():.3f}')

    return onnx_path

# ─── MAIN LOOP: train all commands ───────────────────────────────────
trained_models = []
for i, cmd in enumerate(COMMANDS):
    print(f'\n{"="*70}')
    print(f'Command {i+1}/{len(COMMANDS)}: "{cmd["phrase"]}" → {cmd["model_name"]}')
    print(f'{"="*70}')
    config, config_path = generate_clips_for_command(cmd)
    onnx_path = train_command(cmd, config, config_path)
    trained_models.append((cmd, onnx_path))

    # Download each model as it completes
    from google.colab import files
    files.download(onnx_path)
    print(f'  downloaded: {os.path.basename(onnx_path)}')

print(f'\n{"="*70}')
print(f'DONE. Trained {len(trained_models)} command models.')
print(f'{"="*70}')
for cmd, path in trained_models:
    print(f'  {cmd["model_name"]:20s} → {os.path.basename(path)} ({os.path.getsize(path)/1e3:.0f} KB)')
print(f'\nPlace all .onnx files at:')
print(f'  src-tauri/resources/oww/commands/')

## 7. Export intent mapping JSON

This creates a `command_intents.json` file that NEXUS loads at startup to map each command model to its intent.

In [ ]:
import json, os

intent_map = {}
for cmd, onnx_path in trained_models:
    intent_map[cmd['model_name']] = {
        'phrase': cmd['phrase'],
        'model_file': f'{cmd["model_name"]}.onnx',
        'intent': cmd['intent'],
    }

json_path = '/content/command_intents.json'
with open(json_path, 'w') as f:
    json.dump(intent_map, f, indent=2)
print(f'Wrote {json_path}:')
print(json.dumps(intent_map, indent=2))

from google.colab import files
files.download(json_path)
print(f'\nDownloaded command_intents.json')
print(f'Place at: src-tauri/resources/oww/commands/command_intents.json')

## After Download

Place all downloaded `.onnx` files and `command_intents.json` at:

```
src-tauri/resources/oww/commands/
  ├── open_youtube.onnx
  ├── open_gmail.onnx
  ├── open_chrome.onnx
  ├── open_notepad.onnx
  ├── open_calculator.onnx
  ├── open_spotify.onnx
  ├── open_discord.onnx
  ├── open_github.onnx
  ├── open_vscode.onnx
  ├── open_figma.onnx
  └── command_intents.json
```

NEXUS will load all command models at startup alongside the wake-word model.
When a command model fires, NEXUS executes the mapped intent directly — **no STT needed**.